# Project Title
#### Predicting Retail Product Demand for Inventory Planning
##### A Time-Series Machine Learning Approach Using the Walmart M5 Dataset
# 1. Business Context

Retailers need to anticipate future product demand in order to make better
inventory and operational decisions.

If demand is underestimated, a retailer may experience insufficient stock, missed sales opportunities and lower customer satisfaction. If demand is overestimated, the retailer may hold unnecessary inventory and increase storage or markdown costs.

This project uses the Walmart M5 retail forecasting dataset to investigate
whether historical sales patterns, product characteristics, store information,
calendar effects, special events, SNAP indicators, and selling prices can be
used to predict future product demand.

**The project combines:**
- Exploratory Data Analysis (EDA)
- Time-Series Analysis
- Feature Engineering
- Supervised Machine Learning
- Ensemble Models
- XGBoost
- Time-aware validation
- Hyperparameter tuning
- Model evaluation
- Business interpretation

## 2. Business Problem

Retail demand varies over time due to factors such as:

- historical purchasing patterns
- day of the week
- seasonality
- holidays and special events
- product category
- store location
- selling price
- SNAP-related purchasing conditions

A retailer therefore needs reliable demand forecasts to support planning
decisions.

The business problem addressed in this project is:

> How can historical retail sales and related explanatory variables be used
> to predict future product demand?

The objective is to develop and evaluate machine learning models that can
forecast future unit sales while following a realistic time-series validation
strategy that prevents information from the future from being used to predict
the past.

## 3. Main Research Question

**To what extent can historical demand, temporal seasonality, calendar events, SNAP availability, pricing information, store characteristics, and product-category differences improve short-term retail demand forecasting for inventory planning?**

**Supporting questions:**
- *RQ1 — Temporal patterns*
How do seasonality, calendar events, and recurring demand patterns influence retail product demand?
- *RQ2 — Retail differences*
How does demand vary across stores, states, and product categories?
- *RQ3 — Machine learning*
Can XGBoost improve 28-day demand forecasts compared with a seasonal-naive baseline?


> Can historical sales patterns, pricing information, calendar effects,
> and product/store characteristics be used to accurately predict future
> retail product demand?
Then we'll have supporting questions.
Markdown cell
## Supporting Research Questions

1. How does retail demand change over time?

2. Which product categories, departments, and stores generate the highest
   sales volumes?

3. What seasonal, weekly, calendar, and event-related patterns can be
   identified in retail demand?

4. How are selling prices associated with changes in unit sales?

5. Does adding calendar and price information improve predictions compared
   with using historical sales information alone?

6. Which machine learning model provides the most useful predictive
   performance for this forecasting problem?

7. Which features contribute most strongly to the final XGBoost model?
This gives us a much stronger project story than simply "train XGBoost."







## Step 1 — Import Libraries

The first step is to import the Python libraries required for data loading, manipulation, and analysis.

At this stage, we use **Pandas** and **Numpy** for working with the M5 datasets and **OS** for checking the project directory and file structure.

Additional libraries will be imported later when they are needed for visualization, feature engineering, statistical analysis, and machine learning.


In [1]:
# Step 1: Import libraries

import pandas as pd
import numpy as np
import os

print("Libraries imported successfully.")

Libraries imported successfully.


# 1. Data Acquisition

The M5 Forecasting dataset was downloaded from the Kaggle M5 Forecasting -
Accuracy competition.

The dataset contains historical Walmart retail sales together with
calendar and price information.

The main files required for this project are:

- `sales_train_validation.csv`
- `calendar.csv`
- `sell_prices.csv`

The remaining files are not required for the initial analysis.

## Step 2 — Check the Project Folders

Before loading any data, we verify that the project structure is correctly organized and that the required M5 dataset files are available.

The project separates the raw data from the notebook and source code. This makes the analysis easier to reproduce and helps prevent accidental modification of the original datasets.

The `data` folder should contain:

* `calendar.csv`
* `sales_train_validation.csv`
* `sell_prices.csv`


In [2]:
# Step 2: Check the project folders

data_path = "../data"

print("Files in the data folder:")
print(os.listdir(data_path))

Files in the data folder:
['calendar.csv', 'sales_train_validation.csv', 'sell_prices.csv']


## Step 3 - Check File Sizes

The M5 dataset contains large files, particularly the historical sales and pricing data.

Before loading the complete datasets, we check their file sizes. This helps to understand the computational requirements of the project and avoid unnecessary memory usage.

The sales and pricing files are substantially larger than the calendar file. Therefore, later data-processing steps will use techniques such as **selective loading, data types, and chunked processing** where appropriate.


In [3]:
# Step 3: Check file sizes

for file in os.listdir(data_path):
    file_path = os.path.join(data_path, file)
    size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"{file}: {size_mb:.2f} MB")

calendar.csv: 0.10 MB
sales_train_validation.csv: 114.45 MB
sell_prices.csv: 193.97 MB


## Step 4 — Explore the Calendar Dataset

The M5 calendar dataset contains the daily information needed to understand temporal demand patterns.

It provides:

* dates
* weekday information
* week and month information
* year information
* event and holiday information
* event types
* SNAP availability for California, Texas, and Wisconsin

The calendar will later be combined with the sales data so that the forecasting model can learn how demand changes according to **seasonality, holidays, special events, and SNAP availability**.

Missing values in the event columns are expected because most days do not contain a special event.


In [4]:
# Step 4.0 : Load and inspect the calendar data

# Load the calendar CSV file into a pandas DataFrame
calendar = pd.read_csv("../data/calendar.csv")

# Display the size of the dataset
print("Shape:", calendar.shape)

# Display the first five rows
display(calendar.head())

Shape: (1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [5]:
# step 4.1 : Check the calendar columns

print("Columns:")
print(calendar.columns.tolist())

Columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


In [6]:
# step 4.2 : Check for missing values

calendar.isna().sum()

date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64

## Step 5 — Explore Sales Dataset

The `sales_train_validation.csv` file contains historical daily unit sales for individual products across multiple stores.

The dataset contains:

* product identifiers
* department identifiers
* category identifiers
* store identifiers
* state identifiers
* daily sales from `d_1` to `d_1913`

Because the complete sales dataset is large, an initial sample is loaded to understand its structure without unnecessarily consuming memory.

The complete sales data will be processed later using a more memory-efficient approach.


In [7]:
# Step 5.0 : Inspect the sales data
sales = pd.read_csv("../data/sales_train_validation.csv")

print("sales_train_validation:", sales.shape)
display(sales.head())

sales_train_validation: (30490, 1919)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [8]:
# Step 5.1 : Check the sales data columns

print("Number of columns:", len(sales.columns))
print(sales.columns.tolist())

Number of columns: 1919
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14', 'd_15', 'd_16', 'd_17', 'd_18', 'd_19', 'd_20', 'd_21', 'd_22', 'd_23', 'd_24', 'd_25', 'd_26', 'd_27', 'd_28', 'd_29', 'd_30', 'd_31', 'd_32', 'd_33', 'd_34', 'd_35', 'd_36', 'd_37', 'd_38', 'd_39', 'd_40', 'd_41', 'd_42', 'd_43', 'd_44', 'd_45', 'd_46', 'd_47', 'd_48', 'd_49', 'd_50', 'd_51', 'd_52', 'd_53', 'd_54', 'd_55', 'd_56', 'd_57', 'd_58', 'd_59', 'd_60', 'd_61', 'd_62', 'd_63', 'd_64', 'd_65', 'd_66', 'd_67', 'd_68', 'd_69', 'd_70', 'd_71', 'd_72', 'd_73', 'd_74', 'd_75', 'd_76', 'd_77', 'd_78', 'd_79', 'd_80', 'd_81', 'd_82', 'd_83', 'd_84', 'd_85', 'd_86', 'd_87', 'd_88', 'd_89', 'd_90', 'd_91', 'd_92', 'd_93', 'd_94', 'd_95', 'd_96', 'd_97', 'd_98', 'd_99', 'd_100', 'd_101', 'd_102', 'd_103', 'd_104', 'd_105', 'd_106', 'd_107', 'd_108', 'd_109', 'd_110', 'd_111', 'd_112', 'd_113', 'd_1

## Step 6 - Explore Sell Prices Dataset

The `sell_prices.csv` file contains pricing information for products across stores and weeks.

The main variables are:

* `store_id`
* `item_id`
* `wm_yr_wk`
* `sell_price`

Pricing is an important potential predictor of demand because changes in price may be associated with changes in sales volume.

The prices are recorded at the **store-product-week level**, so they will later need to be aligned with the calendar and aggregated appropriately for our store-category forecasting dataset.


In [9]:
# Step 6.0: Inspect a small sample of the price data

prices = pd.read_csv("../data/sell_prices.csv")


print("Sell_prices shape:", prices.shape)

display(prices.head())

Sell_prices shape: (6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [10]:
# Step 6.1: Check the price data columns
print("Columns:")
print(prices.columns.tolist())

Columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


## Step 7 — Understand the Dataset Structure

Before building the forecasting dataset, we examine the hierarchical structure of the M5 retail data.

The dataset contains several levels of retail information:

**State → Store → Category → Department → Product**

Understanding this hierarchy is important because our forecasting problem will eventually preserve differences between **stores, states, and product categories**.

We therefore investigate:

1. The number of products, departments, categories, stores, and states.
2. The historical date range.
3. The distribution of stores across states.
4. The distribution of products across categories.
5. The distribution of categories across individual stores.

These checks help us understand the scope and structure of the retail network before aggregation and feature engineering.


### Step 7.1 — Product and Retail Hierarchy

We first examine the number of unique products, departments, categories, stores, and states represented in the sales data.

This provides an overview of the retail hierarchy and confirms that the dataset contains multiple geographical and product segments that can contribute to the forecasting problem.


In [11]:
# Step 7.1: Basic sales structure

print("Number of rows in sales data:", len(sales))
print("Unique items:", sales["item_id"].nunique())
print("Unique departments:", sales["dept_id"].nunique())
print("Unique categories:", sales["cat_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Unique states:", sales["state_id"].nunique())

Number of rows in sales data: 30490
Unique items: 3049
Unique departments: 7
Unique categories: 3
Unique stores: 10
Unique states: 3


### Step 7.2 — Product Hierarchy Example

The first few records are examined to understand how individual products are associated with departments, categories, stores, and states.

This confirms the hierarchical relationship between the product and geographical dimensions of the dataset.

For the final forecasting dataset, individual products will be aggregated so that demand can be analyzed at the **store-category-day level**.


In [12]:
# Step 7.2: Inspect product hierarchy

display(
    sales[
        ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
    ]
)

,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA
...,...,...,...,...,...
30485,FOODS_3_823,FOODS_3,FOODS,WI_3,WI
30486,FOODS_3_824,FOODS_3,FOODS,WI_3,WI
30487,FOODS_3_825,FOODS_3,FOODS,WI_3,WI
30488,FOODS_3_826,FOODS_3,FOODS,WI_3,WI


### Step 7.3 — Historical Date Range

We examine the start date, end date, and number of unique dates available in the calendar.

Understanding the historical period is important for time-series forecasting because the model needs sufficient historical observations to learn recurring patterns such as weekly and yearly seasonality.

The M5 calendar covers several years, providing a substantial historical period for identifying these patterns.


In [13]:
# Step 7.3: Check the calendar date range

calendar["date"] = pd.to_datetime(calendar["date"])

print("Start date:", calendar["date"].min())
print("End date:", calendar["date"].max())
print("Number of days:", calendar["date"].nunique())

Start date: 2011-01-29 00:00:00
End date: 2016-06-19 00:00:00
Number of days: 1969


### Step 7.4 — Category and Department Distribution

We examine how products are distributed across categories and departments.

The M5 dataset contains three main product categories:

* **FOODS**
* **HOUSEHOLD**
* **HOBBIES**

The categories contain different numbers of products and departments. These differences are important because demand behavior can vary substantially between product categories.

Rather than modeling only one category, this project will retain all three categories so that category-level differences can be incorporated into the forecasting analysis.

I will use `usecol` so that pandas reads only the needed **six columns** rather the the entire sales dataset. 


In [14]:
# Step 7.4a: Load only product/store information

sales_info = pd.read_csv(
    "../data/sales_train_validation.csv",
    usecols=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ]
)

print("Shape:", sales_info.shape)

Shape: (30490, 6)


In [15]:
# step 7.4b: count the frequencies of each unique categories and departments
print("Categories:")
print(sales_info["cat_id"].value_counts())

print("Departments:")
print(sales_info["dept_id"].value_counts())

Categories:
cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64
Departments:
dept_id
FOODS_3        8230
HOUSEHOLD_1    5320
HOUSEHOLD_2    5150
HOBBIES_1      4160
FOODS_2        3980
FOODS_1        2160
HOBBIES_2      1490
Name: count, dtype: int64


### Step 7.5 — Store and State Distribution

We examine the distribution of products across stores and states.

The M5 dataset contains:

* **10 stores**
* **3 states:** California, Texas, and Wisconsin

Each store belongs to one state, allowing the forecasting model to incorporate both **store-level and state-level differences**.

We also examine the distribution of product categories across stores to confirm that the three main categories are represented throughout the retail network.

These geographical differences will later be retained as features in the forecasting dataset.


In [16]:
# Step 7.5a: Inspect stores and states

print("Stores:")
print(sales_info["store_id"].value_counts().sort_index())

print("\nStates:")
print(sales_info["state_id"].value_counts().sort_index())

Stores:
store_id
CA_1    3049
CA_2    3049
CA_3    3049
CA_4    3049
TX_1    3049
TX_2    3049
TX_3    3049
WI_1    3049
WI_2    3049
WI_3    3049
Name: count, dtype: int64

States:
state_id
CA    12196
TX     9147
WI     9147
Name: count, dtype: int64


In [17]:
# Step 7.5b: Check category distribution by store

category_store_counts = pd.crosstab(
    sales_info["store_id"],
    sales_info["cat_id"]
)

display(category_store_counts)

cat_id,FOODS,HOBBIES,HOUSEHOLD
store_id,,,
CA_1,1437,565,1047
CA_2,1437,565,1047
CA_3,1437,565,1047
CA_4,1437,565,1047
TX_1,1437,565,1047
TX_2,1437,565,1047
TX_3,1437,565,1047
WI_1,1437,565,1047
WI_2,1437,565,1047


Step 7.5 output shows a perfect balanced dataset across all stores level.
Every store contains exactly:

- 3,049 product/store combinations
- 1,437 FOODS
- 1,047 HOUSEHOLD
- 565 HOBBIES

## Step 8 — Explore Calendar Events and Holidays

Retail demand can change significantly around holidays, cultural occasions, religious events, and sporting events. Therefore, calendar events are an important part of the forecasting problem.

The M5 calendar provides event names and event types rather than requiring the dates to be manually created.

The dataset includes events such as:

* Christmas
* Easter
* Thanksgiving
* Valentine's Day
* Mother's Day
* Father's Day
* Independence Day
* Halloween
* Super Bowl
* NBA Finals
* Ramadan
* Eid al-Fitr
* Eid al-Adha
* Lent
* Orthodox Easter
* other national, cultural, religious, and sporting events

The event information is provided through two event columns (`event_name_1` and `event_name_2`), because some dates can contain more than one event.

In total, the calendar contains **162 days with event information**.

These variables will later be incorporated into the forecasting dataset so that the XGBoost model can learn whether demand patterns change around special occasions.

### Why events matter for a 28-day forecast

The forecast horizon is 28 days, but the model is trained using the historical period rather than only the previous 28 days.

This allows the model to learn recurring patterns from previous years. For example, historical Christmas periods can help the model understand demand behavior around a future Christmas period.

The same principle applies to Easter, Thanksgiving, Ramadan, Valentine's Day, and other recurring events.

In addition to event information, the final forecasting dataset will incorporate **SNAP availability and pricing information**, allowing the model to consider multiple factors that may influence retail demand.


### Step 8.1 — Event Types

The M5 calendar classifies events into several types:

* **Sporting**
* **Cultural**
* **National**
* **Religious**

These classifications provide an additional level of information beyond the event name itself.

For example, the model can potentially learn different demand patterns associated with national holidays compared with sporting or religious events.


In [18]:
# Step 8a: Explore calendar events and holidays

print("Event names in event_name_1:")
print(calendar["event_name_1"].dropna().unique())

print("\nEvent names in event_name_2:")
print(calendar["event_name_2"].dropna().unique())

print("\nEvent types in event_type_1:")
print(calendar["event_type_1"].dropna().unique())

print("\nEvent types in event_type_2:")
print(calendar["event_type_2"].dropna().unique())

Event names in event_name_1:
['SuperBowl' 'ValentinesDay' 'PresidentsDay' 'LentStart' 'LentWeek2'
 'StPatricksDay' 'Purim End' 'OrthodoxEaster' 'Pesach End' 'Cinco De Mayo'
 "Mother's day" 'MemorialDay' 'NBAFinalsStart' 'NBAFinalsEnd'
 "Father's day" 'IndependenceDay' 'Ramadan starts' 'Eid al-Fitr'
 'LaborDay' 'ColumbusDay' 'Halloween' 'EidAlAdha' 'VeteransDay'
 'Thanksgiving' 'Christmas' 'Chanukah End' 'NewYear' 'OrthodoxChristmas'
 'MartinLutherKingDay' 'Easter']

Event names in event_name_2:
['Easter' 'Cinco De Mayo' 'OrthodoxEaster' "Father's day"]

Event types in event_type_1:
['Sporting' 'Cultural' 'National' 'Religious']

Event types in event_type_2:
['Cultural' 'Religious']


### Step 8.2 — SNAP Availability

The M5 calendar also provides state-specific SNAP availability indicators for:

* California (`snap_CA`)
* Texas (`snap_TX`)
* Wisconsin (`snap_WI`)

SNAP availability may influence purchasing behavior and therefore represents a potentially useful external demand feature.

Because SNAP information is state-specific, the final forecasting dataset will select the appropriate SNAP indicator based on each store's state.

This allows the model to distinguish between SNAP conditions in California, Texas, and Wisconsin rather than treating SNAP as a single national variable.


In [19]:
# Step 8b: Count event days

event_days = calendar[
    calendar["event_name_1"].notna() |
    calendar["event_name_2"].notna()
]

print("Number of event days:", len(event_days))

display(
    event_days[
        [
            "date",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ].head(20)
)

Number of event days: 162


,date,event_name_1,event_type_1,event_name_2,event_type_2
8,2011-02-06,SuperBowl,Sporting,NaN,NaN
16,2011-02-14,ValentinesDay,Cultural,NaN,NaN
23,2011-02-21,PresidentsDay,National,NaN,NaN
39,2011-03-09,LentStart,Religious,NaN,NaN
46,2011-03-16,LentWeek2,Religious,NaN,NaN
47,2011-03-17,StPatricksDay,Cultural,NaN,NaN
50,2011-03-20,Purim End,Religious,NaN,NaN
85,2011-04-24,OrthodoxEaster,Religious,Easter,Cultural
87,2011-04-26,Pesach End,Religious,NaN,NaN
96,2011-05-05,Cinco De Mayo,Cultural,NaN,NaN


## Step 9 — Building the Forecasting Dataset

The original M5 sales dataset contains daily sales for 30,490 individual products across 1,913 historical days. Using the complete product-level dataset would create more than 58 million product-day observations and would require substantially more memory and processing time.

For this project, the sales data will therefore be aggregated to the **store-category-day level**.

This means that each observation in the modeling dataset will represent the total daily demand for one product category within one store.

The resulting dataset will contain:

* **10 stores**
* **3 product categories:** FOODS, HOUSEHOLD, and HOBBIES
* **3 states:** California, Texas, and Wisconsin
* **1,913 historical days**
* approximately **57,390 store-category-day observations**

This level of aggregation allows the model to retain important differences between stores, states, and product categories while keeping the dataset computationally manageable.

The forecasting dataset will later be combined with calendar, event, SNAP, and pricing information. Historical demand features such as lagged sales and rolling averages will then be created for the XGBoost forecasting model.

### Forecasting structure

**Individual products → Store × Category → Daily demand**

This approach allows the project to investigate whether historical demand patterns, seasonality, special events, SNAP availability, pricing changes, store characteristics, and category differences can improve **28-day retail demand forecasts for inventory planning**.


### Step 9.1 — Identifying the Daily Sales Columns

The M5 sales file stores historical demand in separate columns for each day, from `d_1` to `d_1913`.

Before aggregating the sales data, we identify these daily sales columns programmatically rather than manually listing them. This makes the data preparation process more reliable and easier to reproduce.

The 1,913 daily columns represent the historical demand period that will be used to learn recurring patterns such as:

* weekly seasonality
* yearly seasonality
* holiday and event effects
* changes in demand over time
* store-level differences
* category-level differences

The daily columns will later be transformed from the original wide format into a modeling-friendly time-series structure.


In [20]:
# Step 9.1: Identify the daily sales columns

sales_columns = pd.read_csv(
    "../data/sales_train_validation.csv",
    nrows=0
).columns.tolist()

daily_columns = [
    col for col in sales_columns
    if col.startswith("d_")
]

print("Number of daily sales columns:", len(daily_columns))
print("First 5:", daily_columns[:5])
print("Last 5:", daily_columns[-5:])

Number of daily sales columns: 1913
First 5: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last 5: ['d_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


### Step 9.2 — Efficiently Aggregate Daily Sales

The original sales dataset contains 30,490 individual products and 1,913 daily sales columns. Converting the complete dataset directly into a long format would create more than 58 million product-day observations and could consume a large amount of memory.

To make the data preparation more efficient, the sales file will be processed in **chunks**.

For each chunk, product-level sales will be aggregated by:

* `store_id`
* `state_id`
* `cat_id`

The result will represent total daily demand for each **store × category** combination.

The aggregated data will then be combined across all chunks.

This approach preserves the geographical and product-category differences needed for the forecasting model while avoiding the memory cost of creating a very large product-level time-series dataset.

The final output will contain approximately:

**10 stores × 3 categories × 1,913 days = 57,390 observations.**


In [21]:
# Step 9.2: Aggregate product-level sales by store and category


# Columns needed for the aggregation
group_columns = ["store_id", "state_id", "cat_id"]

# Store the aggregated results from each chunk
aggregated_chunks = []

# Define data types for daily sales to reduce memory usage
sales_dtype = {col: "uint16" for col in daily_columns}

# Read the sales file in chunks
for chunk in pd.read_csv(
    "../data/sales_train_validation.csv",
    usecols=group_columns + daily_columns,
    dtype=sales_dtype,
    chunksize=2000
):
    
    # Aggregate product-level sales within each store and category
    chunk_aggregated = (
        chunk
        .groupby(group_columns)[daily_columns]
        .sum()
    )
    
    aggregated_chunks.append(chunk_aggregated)

# Combine the aggregated chunks
sales_aggregated = pd.concat(aggregated_chunks)

# Aggregate again because the same store-category combination
# may appear in more than one chunk
sales_aggregated = (
    sales_aggregated
    .groupby(group_columns)[daily_columns]
    .sum()
)

print("Aggregated dataset shape:", sales_aggregated.shape)
display(sales_aggregated.head())

Aggregated dataset shape: (30, 1913)


d_1   d_2   d_3   d_4   d_5   d_6   d_7   d_8  \
store_id state_id cat_id                                                      
CA_1     CA       FOODS      3239  3137  2008  2258  2032  2407  2693  3962   
                  HOBBIES     556   498   415   392   268   467   346   769   
                  HOUSEHOLD   542   520   393   401   330   402   411   706   
CA_2     CA       FOODS      2193  1921  1289  1540  1278  1494  1785  2385   
                  HOBBIES     538   397   368   350   296   391   316   413   

                              d_9  d_10  ...  d_1904  d_1905  d_1906  d_1907  \
store_id state_id cat_id                 ...                                   
CA_1     CA       FOODS      3418  2446  ...    2706    3481    4004    2886   
                  HOBBIES     432   384  ...     401     722     655     542   
                  HOUSEHOLD   490   327  ...     875    1234    1295     917   
CA_2     CA       FOODS      1827  1355  ...    2891    3423    3729    2625   
                  HOBBIES     452   403  ...     483     486     483     320   

                             d_1908  d_1909  d_1910  d_1911  d_1912  d_1913  
store_id state_id cat_id                                                     
CA_1     CA       FOODS        2488    2490    2533    2960    3655    4053  
                  HOBBIES       495     363     410     527     670     714  
                  HOUSEHOLD     810     869     766     900    1252    1346  
CA_2     CA       FOODS        2410    2506    2261    2871    3760    4012  
                  HOBBIES       368     344     271     485     542     507  

[5 rows x 1913 columns]

### Step 9.3 — Reshape the Aggregated Sales Data

The aggregated sales data is currently in **wide format**, where each day is stored as a separate column (`d_1`, `d_2`, ..., `d_1913`).

Although this format is useful for efficient aggregation, it is not ideal for time-series analysis or machine learning.

We will therefore transform the data into **long format**, where each row represents one observation for a specific:

* store
* state
* product category
* day
* demand value

The resulting structure will be:

| store_id | state_id | cat_id | d   | demand |
| -------- | -------- | ------ | --- | -----: |
| CA_1     | CA       | FOODS  | d_1 |   3239 |
| CA_1     | CA       | FOODS  | d_2 |   3137 |
| CA_1     | CA       | FOODS  | d_3 |   2008 |

With 30 store-category combinations and 1,913 days, this will produce approximately **57,390 observations**.

This long-format structure will allow us to merge the sales data with the M5 calendar and later create time-series features such as lagged demand and rolling averages.


In [22]:
# Step 9.3: Convert sales data from wide format to long format

sales_long = (
    sales_aggregated
    .reset_index()
    .melt(
        id_vars=["store_id", "state_id", "cat_id"],
        value_vars=daily_columns,
        var_name="d",
        value_name="demand"
    )
)

print("Long-format dataset shape:", sales_long.shape)

display(sales_long.head(10))

Long-format dataset shape: (57390, 5)


,store_id,state_id,cat_id,d,demand
0,CA_1,CA,FOODS,d_1,3239
1,CA_1,CA,HOBBIES,d_1,556
2,CA_1,CA,HOUSEHOLD,d_1,542
3,CA_2,CA,FOODS,d_1,2193
4,CA_2,CA,HOBBIES,d_1,538
5,CA_2,CA,HOUSEHOLD,d_1,763
6,CA_3,CA,FOODS,d_1,3446
7,CA_3,CA,HOBBIES,d_1,550
8,CA_3,CA,HOUSEHOLD,d_1,743
9,CA_4,CA,FOODS,d_1,1223


### Step 9.4 — Merge Sales with Calendar Data

The sales dataset currently identifies each day using the M5 day identifier (`d_1` to `d_1913`). However, the forecasting model needs actual calendar information to learn temporal and seasonal patterns.

The M5 calendar dataset uses the same `d` identifiers, which allows the sales data to be matched directly with the calendar.

We will merge the two datasets to add:

* the actual date
* weekday
* week number
* month
* year
* event names
* event types
* SNAP indicators
* M5 week identifier (`wm_yr_wk`)

This creates the foundation for our time-series feature engineering.

In particular, this step allows special occasions such as **Christmas, Easter, Thanksgiving, Ramadan, and other events** to become variables that can be used in the forecasting model.

The merge is performed as a **left join**, ensuring that every sales observation remains in the dataset.


In [23]:
# Step 9.4: Merge sales data with the calendar

sales_long = sales_long.merge(
    calendar,
    on="d",
    how="left"
)

print("Dataset shape after calendar merge:", sales_long.shape)

display(
    sales_long[
        [
            "store_id",
            "state_id",
            "cat_id",
            "d",
            "date",
            "demand",
            "weekday",
            "wday",
            "month",
            "year",
            "wm_yr_wk"
        ]
    ].head(10)
)

Dataset shape after calendar merge: (57390, 18)


,store_id,state_id,cat_id,d,date,demand,weekday,wday,month,year,wm_yr_wk
0,CA_1,CA,FOODS,d_1,2011-01-29,3239,Saturday,1,1,2011,11101
1,CA_1,CA,HOBBIES,d_1,2011-01-29,556,Saturday,1,1,2011,11101
2,CA_1,CA,HOUSEHOLD,d_1,2011-01-29,542,Saturday,1,1,2011,11101
3,CA_2,CA,FOODS,d_1,2011-01-29,2193,Saturday,1,1,2011,11101
4,CA_2,CA,HOBBIES,d_1,2011-01-29,538,Saturday,1,1,2011,11101
5,CA_2,CA,HOUSEHOLD,d_1,2011-01-29,763,Saturday,1,1,2011,11101
6,CA_3,CA,FOODS,d_1,2011-01-29,3446,Saturday,1,1,2011,11101
7,CA_3,CA,HOBBIES,d_1,2011-01-29,550,Saturday,1,1,2011,11101
8,CA_3,CA,HOUSEHOLD,d_1,2011-01-29,743,Saturday,1,1,2011,11101
9,CA_4,CA,FOODS,d_1,2011-01-29,1223,Saturday,1,1,2011,11101


In [24]:
# Check whether the calendar merge created missing dates

print("Missing dates:", sales_long["date"].isna().sum())
print("Number of unique dates:", sales_long["date"].nunique())

Missing dates: 0
Number of unique dates: 1913


In [25]:
# verify that our state-specific SNAP columns are present
print("SNAP columns:")
print(
    sales_long[
        ["snap_CA", "snap_TX", "snap_WI"]
    ].head()
)

SNAP columns:
   snap_CA  snap_TX  snap_WI
0        0        0        0
1        0        0        0
2        0        0        0
3        0        0        0
4        0        0        0


### Step 9.5 — Create the State-Specific SNAP Feature

SNAP availability in the M5 dataset is provided separately for California, Texas, and Wisconsin through the columns `snap_CA`, `snap_TX`, and `snap_WI`.

Because each store belongs to one state, we can create a single `snap` variable that selects the appropriate SNAP indicator for each observation.

For example:

* A California store uses `snap_CA`
* A Texas store uses `snap_TX`
* A Wisconsin store uses `snap_WI`

This creates one consistent feature that represents whether SNAP was active for the relevant store and date.

The three original state-specific columns will be retained temporarily for verification and can be removed later if they are no longer needed.


In [26]:
# Step 9.5: Create one state-specific SNAP feature

sales_long["snap"] = np.select(
    [
        sales_long["state_id"] == "CA",
        sales_long["state_id"] == "TX",
        sales_long["state_id"] == "WI"
    ],
    [
        sales_long["snap_CA"],
        sales_long["snap_TX"],
        sales_long["snap_WI"]
    ],
    default=0
)

print("SNAP feature created successfully.")

display(
    sales_long[
        [
            "store_id",
            "state_id",
            "date",
            "snap_CA",
            "snap_TX",
            "snap_WI",
            "snap"
        ]
    ].head(15)
)

SNAP feature created successfully.


,store_id,state_id,date,snap_CA,snap_TX,snap_WI,snap
0,CA_1,CA,2011-01-29,0,0,0,0
1,CA_1,CA,2011-01-29,0,0,0,0
2,CA_1,CA,2011-01-29,0,0,0,0
3,CA_2,CA,2011-01-29,0,0,0,0
4,CA_2,CA,2011-01-29,0,0,0,0
5,CA_2,CA,2011-01-29,0,0,0,0
6,CA_3,CA,2011-01-29,0,0,0,0
7,CA_3,CA,2011-01-29,0,0,0,0
8,CA_3,CA,2011-01-29,0,0,0,0
9,CA_4,CA,2011-01-29,0,0,0,0


In [27]:
# Verify SNAP values

print("Unique SNAP values:", sales_long["snap"].unique())

print("\nSNAP distribution:")
print(sales_long["snap"].value_counts().sort_index())

Unique SNAP values: [0 1]

SNAP distribution:
snap
0    38490
1    18900
Name: count, dtype: int64


Derived feature: snap was created from the original state-specific M5 SNAP indicators (snap_CA, snap_TX, and snap_WI) according to each store's state.

### Step 9.6 — Verify Store and State Structure

The M5 dataset contains 10 stores distributed across three states:

* California
* Texas
* Wisconsin

Since state-specific information such as SNAP availability depends on the store's location, it is important to verify that each store is consistently associated with one state.

This check ensures that the state-level features used later in the forecasting model are correctly aligned with each store.


In [28]:
# Step 9.6: Verify each store belongs to one state

store_state_check = (
    sales_long[["store_id", "state_id"]]
    .drop_duplicates()
    .sort_values(["state_id", "store_id"])
)

display(store_state_check)

,store_id,state_id
0,CA_1,CA
3,CA_2,CA
6,CA_3,CA
9,CA_4,CA
12,TX_1,TX
15,TX_2,TX
18,TX_3,TX
21,WI_1,WI
24,WI_2,WI
27,WI_3,WI


## Step 10 — Integrate Pricing Information

Price is an important potential driver of retail demand. The M5 dataset provides weekly selling prices for individual products at each store.

Unlike the sales data, which contains daily observations, the price data is structured at the:

**store × product × week**

level.

The price information therefore cannot be merged directly with the forecasting dataset using the calendar date alone.

Instead, the price data will be connected using:

* `store_id`
* `item_id`
* `wm_yr_wk`

Because the forecasting dataset is aggregated to the **store × category × day** level, individual product prices must also be aggregated to the store-category level.

For the initial forecasting dataset, we will calculate the **average listed selling price** for each store, category, and week.

We will then merge this weekly price information onto the daily forecasting data through `store_id`, `cat_id`, and `wm_yr_wk`.

In addition to the price level, we will later create a **week-over-week price change feature**. This will allow the model to learn whether changes in pricing are associated with changes in demand.

### Important modeling consideration

The M5 price data is weekly, while demand is daily. Therefore, the same weekly price value will apply to the corresponding daily observations within that week.

This is appropriate for the structure of the source data and avoids artificially creating daily price changes that are not present in the dataset.


### Step 10.1 — Create the Item-to-Category Mapping

The pricing dataset identifies products using `item_id`, while our forecasting dataset uses product categories.

To connect the two datasets, we first create a lookup table linking every `item_id` to its corresponding category.

The mapping is obtained from the sales metadata, which contains the relationship between products and categories.

This lookup will allow us to transform the product-level pricing data into store-category-level pricing information without loading the complete sales dataset again.


In [29]:
# Step 10.1: Create an item-to-category mapping

item_category = pd.read_csv(
    "../data/sales_train_validation.csv",
    usecols=["item_id", "cat_id"]
).drop_duplicates()

print("Number of unique items:", item_category["item_id"].nunique())
print("Number of item-category mappings:", len(item_category))

display(item_category.head())

Number of unique items: 3049
Number of item-category mappings: 3049


,item_id,cat_id
0,HOBBIES_1_001,HOBBIES
1,HOBBIES_1_002,HOBBIES
2,HOBBIES_1_003,HOBBIES
3,HOBBIES_1_004,HOBBIES
4,HOBBIES_1_005,HOBBIES


### Step 10.2 — Aggregate Weekly Prices

The `sell_prices.csv` file contains weekly prices for individual products across stores.

Because the file is relatively large, it will be processed in chunks rather than loaded completely into memory.

Each price record will first be matched to its product category using the item-to-category mapping created in Step 10.1.

The prices will then be aggregated by:

* `store_id`
* `cat_id`
* `wm_yr_wk`

This produces a weekly average listed price for each store-category combination.

The weekly average is calculated from total price sums divided by total valid price records, avoiding unequal weighting of chunks.

The resulting table will be substantially smaller than the original price dataset and can be safely merged with the 57,390-row forecasting dataset.


In [30]:
# Step 10.2 - weekly price aggregation

price_chunks = []

for chunk in pd.read_csv(
    "../data/sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    chunksize=100000
):

    chunk = chunk.merge(
        item_category,
        on="item_id",
        how="left",
        validate="many_to_one"
    )

    chunk_summary = (
        chunk.groupby(
            ["store_id", "cat_id", "wm_yr_wk"],
            as_index=False
        )
        .agg(
            price_sum=("sell_price", "sum"),
            price_count=("sell_price", "count")
        )
    )

    price_chunks.append(chunk_summary)

price_summary = pd.concat(
    price_chunks,
    ignore_index=True
)

price_summary = (
    price_summary.groupby(
        ["store_id", "cat_id", "wm_yr_wk"],
        as_index=False
    )
    .agg(
        price_sum=("price_sum", "sum"),
        price_count=("price_count", "sum")
    )
)

assert (price_summary["price_count"] > 0).all()

price_summary["avg_sell_price"] = (
    price_summary["price_sum"] /
    price_summary["price_count"]
)

price_aggregated = price_summary[
    ["store_id", "cat_id", "wm_yr_wk", "avg_sell_price"]
].copy()

assert not price_aggregated.duplicated(
    ["store_id", "cat_id", "wm_yr_wk"]
).any()

print("Price dataset shape:", price_aggregated.shape)
display(price_aggregated.head(10))

Price dataset shape: (8460, 4)


,store_id,cat_id,wm_yr_wk,avg_sell_price
0,CA_1,FOODS,11101,3.149220
1,CA_1,FOODS,11102,3.153394
2,CA_1,FOODS,11103,3.129630
3,CA_1,FOODS,11104,3.123212
4,CA_1,FOODS,11105,3.125868
5,CA_1,FOODS,11106,3.111022
6,CA_1,FOODS,11107,3.117528
7,CA_1,FOODS,11108,3.126890
8,CA_1,FOODS,11109,3.122738
9,CA_1,FOODS,11110,3.135642


### 10.2 Aggregate Weekly Selling Prices by Store and Category

**Objective:** Calculate the average weekly selling price for each Walmart store and product category.

The original `sell_prices.csv` dataset contains weekly selling prices at the store-item level.

To align pricing information with our store-category demand dataset, individual item prices are aggregated by `store_id`, `cat_id`, and `wm_yr_wk`.

Because the dataset is processed in chunks to manage memory usage, the aggregation first calculates the sum of selling prices and the number of valid price observations within each chunk.

These intermediate results are combined to calculate the final weighted average:

\[
\text{Average Selling Price} = \frac{\text{Total Selling Price}}{\text{Number of Valid Price Observations}}
\]

This approach ensures that each original price observation contributes equally, regardless of chunk size.

The resulting dataset contains **8,460 store-category-week records** with four columns:

- `store_id`
- `cat_id`
- `wm_yr_wk`
- `avg_sell_price`

These aggregated prices will subsequently be used to construct pricing features and investigate relationships between selling prices and retail demand.

### Step 10.3 — Merge Weekly Prices with the Forecasting Dataset

The aggregated price dataset contains the average listed selling price for each **store, product category, and week**.

Our forecasting dataset (`sales_long`) is at the **store-category-day** level. Therefore, the weekly price information can be connected to the daily demand data using three keys:

* `store_id` — identifies the store
* `cat_id` — identifies the product category
* `wm_yr_wk` — identifies the M5 calendar week

A left join is used so that all demand observations are retained, even if a corresponding price observation is unavailable.

This creates a daily forecasting dataset containing both **demand** and **pricing information**.


In [31]:
# Step 10.3: Merge weekly prices into the daily forecasting dataset

sales_long = sales_long.merge(
    price_aggregated,
    on=["store_id", "cat_id", "wm_yr_wk"],
    how="left"
)

print("Dataset shape after price merge:", sales_long.shape)

print(
    "Missing average selling prices:",
    sales_long["avg_sell_price"].isna().sum()
)

display(
    sales_long[
        [
            "store_id",
            "state_id",
            "cat_id",
            "date",
            "wm_yr_wk",
            "demand",
            "avg_sell_price"
        ]
    ].head(15)
)

Dataset shape after price merge: (57390, 20)
Missing average selling prices: 0


,store_id,state_id,cat_id,date,wm_yr_wk,demand,avg_sell_price
0,CA_1,CA,FOODS,2011-01-29,11101,3239,3.149220
1,CA_1,CA,HOBBIES,2011-01-29,11101,556,4.468814
2,CA_1,CA,HOUSEHOLD,2011-01-29,11101,542,5.416099
3,CA_2,CA,FOODS,2011-01-29,11101,2193,2.876585
4,CA_2,CA,HOBBIES,2011-01-29,11101,538,4.606471
5,CA_2,CA,HOUSEHOLD,2011-01-29,11101,763,5.560200
6,CA_3,CA,FOODS,2011-01-29,11101,3446,3.041969
7,CA_3,CA,HOBBIES,2011-01-29,11101,550,4.586119
8,CA_3,CA,HOUSEHOLD,2011-01-29,11101,743,5.614755
9,CA_4,CA,FOODS,2011-01-29,11101,1223,2.953270


### Step 10.4 — Check Price Coverage

Before creating price-related features, we need to assess how completely the weekly price information covers the forecasting dataset.

This is important because missing price values could occur when a product or category does not have an available price record for a particular store-week.

We will calculate:

1. The number of missing price observations.
2. The percentage of observations with missing prices.
3. The minimum and maximum selling prices.
4. The number of unique store-category combinations represented in the price data.

This quality check will help determine whether any additional treatment of missing prices is necessary.


In [32]:
# Step 10.4: Check price coverage

missing_prices = sales_long["avg_sell_price"].isna().sum()
total_rows = len(sales_long)

missing_percentage = (missing_prices / total_rows) * 100

print("Total observations:", total_rows)
print("Missing price observations:", missing_prices)
print(f"Missing price percentage: {missing_percentage:.2f}%")

print("\nPrice range:")
print("Minimum price:", sales_long["avg_sell_price"].min())
print("Maximum price:", sales_long["avg_sell_price"].max())

print(
    "\nUnique store-category combinations:",
    sales_long[["store_id", "cat_id"]].drop_duplicates().shape[0]
)

Total observations: 57390
Missing price observations: 0
Missing price percentage: 0.00%

Price range:
Minimum price: 2.759773371104816
Maximum price: 5.841173708920188



Unique store-category combinations: 30


In [33]:
# Check price coverage by store and category

price_coverage = (
    sales_long
    .groupby(["store_id", "cat_id"])["avg_sell_price"]
    .agg(
        observations="count",
        missing="size"
    )
)

price_coverage["missing"] = (
    price_coverage["missing"] - price_coverage["observations"]
)

price_coverage["missing_pct"] = (
    price_coverage["missing"]
    / (price_coverage["observations"] + price_coverage["missing"])
    * 100
)

display(price_coverage)

observations  missing  missing_pct
store_id cat_id                                       
CA_1     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
CA_2     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
CA_3     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
CA_4     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
TX_1     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
TX_2     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
TX_3     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
WI_1     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
WI_2     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0
WI_3     FOODS              1913        0          0.0
         HOBBIES            1913        0          0.0
         HOUSEHOLD          1913        0          0.0


### Step 10.5 Calculate Weekly Price Changes

**Objective:** Measure how average selling prices change from one week to the next for each Walmart store and product category.

Two features are created:

- **price_change:** The difference between the current week's average selling price and the previous week's average selling price.
- **price_change_pct:** The relative change in average selling price compared with the previous week.

These features help us investigate relationships between pricing fluctuations and retail demand.

The first week of each store-category combination will have missing price-change values because no previous observation is available.
The forecasting dataset currently contains the average selling price for each store-category-week.

To capture pricing dynamics, we will create two additional features:

* **`price_change`** — the absolute change in average price from the previous week.
* **`price_change_pct`** — the percentage change in average price from the previous week.

These features may help the model identify whether demand changes are associated with recent price increases or decreases.

The calculation is performed separately for each **store-category combination** and is based on weekly observations.


In [34]:
# Step 10.5 — Calculate Weekly Price Changes

# Sort observations chronologically within each store-category
price_aggregated = (
    price_aggregated
    .sort_values(["store_id", "cat_id", "wm_yr_wk"])
    .reset_index(drop=True)
)

# Calculate absolute weekly price change
price_aggregated["price_change"] = (
    price_aggregated
    .groupby(["store_id", "cat_id"])["avg_sell_price"]
    .diff()
)

# Calculate percentage weekly price change
price_aggregated["price_change_pct"] = (
    price_aggregated
    .groupby(["store_id", "cat_id"])["avg_sell_price"]
    .pct_change(fill_method=None)
)

# Validate the output
assert len(price_aggregated) == 8460
assert not price_aggregated.duplicated(
    ["store_id", "cat_id", "wm_yr_wk"]
).any()

print("Dataset shape:", price_aggregated.shape)

display(price_aggregated.head(15))

Dataset shape: (8460, 6)


,store_id,cat_id,wm_yr_wk,avg_sell_price,price_change,price_change_pct
0,CA_1,FOODS,11101,3.149220,NaN,NaN
1,CA_1,FOODS,11102,3.153394,0.004174,0.001325
2,CA_1,FOODS,11103,3.129630,-0.023764,-0.007536
3,CA_1,FOODS,11104,3.123212,-0.006418,-0.002051
4,CA_1,FOODS,11105,3.125868,0.002656,0.000851
5,CA_1,FOODS,11106,3.111022,-0.014846,-0.004749
6,CA_1,FOODS,11107,3.117528,0.006505,0.002091
7,CA_1,FOODS,11108,3.126890,0.009363,0.003003
8,CA_1,FOODS,11109,3.122738,-0.004152,-0.001328
9,CA_1,FOODS,11110,3.135642,0.012903,0.004132


#### Interpretation — Weekly Price Changes

The weekly price aggregation and price-change calculations were successfully completed for 8,460 store-category-week observations.

The resulting dataset contains average selling prices, absolute weekly price changes, and relative weekly price changes.

Initial observations show both positive and negative price movements, indicating that average selling prices fluctuate over time.

The first observation within each store-category combination contains missing price-change values because no preceding week is available.

These features provide additional information for exploring the relationship between pricing and retail demand and for subsequent machine learning feature engineering.

### 10.6 Merge Weekly Price Changes with Daily Demand

**Objective:** Integrate weekly pricing changes into the daily store-category demand dataset.

The weekly pricing features are merged with the daily sales observations using `store_id`, `cat_id`, and `wm_yr_wk`.

A many-to-one merge is used because multiple daily observations correspond to the same store-category-week combination.

The resulting dataset combines:

- Daily retail demand
- Average weekly selling price
- Absolute weekly price change
- Percentage weekly price change

Validation checks ensure that the merge preserves the original number of daily observations and retains the demand column.

These additional pricing features will support exploratory data analysis and subsequent machine learning modeling.


In [40]:
# Step 10.6 — Merge price-change features into the daily dataset

# Select weekly price-change features
price_features = price_aggregated[
    [
        "store_id",
        "cat_id",
        "wm_yr_wk",
        "price_change",
        "price_change_pct"
    ]
].copy()

# Store original row count
rows_before = len(sales_long)

# Ensure we are not merging duplicate feature columns
assert "price_change" not in sales_long.columns
assert "price_change_pct" not in sales_long.columns

# Merge price-change features into daily demand data
sales_long = sales_long.merge(
    price_features,
    on=["store_id", "cat_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

# Validate merge
assert len(sales_long) == rows_before
assert "demand" in sales_long.columns
assert "avg_sell_price" in sales_long.columns

print(" Completed successfully.")
print("Dataset shape:", sales_long.shape)
print("Missing demand values:", sales_long["demand"].isna().sum())

display(
    sales_long[
        [
            "store_id",
            "cat_id",
            "date",
            "avg_sell_price",
            "price_change",
            "price_change_pct",
            "demand"
        ]
    ].head(15)
)

 Completed successfully.
Dataset shape: (57390, 22)
Missing demand values: 0


,store_id,cat_id,date,avg_sell_price,price_change,price_change_pct,demand
0,CA_1,FOODS,2011-01-29,3.149220,NaN,NaN,3239
1,CA_1,HOBBIES,2011-01-29,4.468814,NaN,NaN,556
2,CA_1,HOUSEHOLD,2011-01-29,5.416099,NaN,NaN,542
3,CA_2,FOODS,2011-01-29,2.876585,NaN,NaN,2193
4,CA_2,HOBBIES,2011-01-29,4.606471,NaN,NaN,538
5,CA_2,HOUSEHOLD,2011-01-29,5.560200,NaN,NaN,763
6,CA_3,FOODS,2011-01-29,3.041969,NaN,NaN,3446
7,CA_3,HOBBIES,2011-01-29,4.586119,NaN,NaN,550
8,CA_3,HOUSEHOLD,2011-01-29,5.614755,NaN,NaN,743
9,CA_4,FOODS,2011-01-29,2.953270,NaN,NaN,1223


#### Interpretation — Integration of Pricing Features

The weekly pricing features were successfully integrated into the daily store-category demand dataset.

The resulting dataset contains **57,390 observations and 22 columns**, with *no missing demand values.*

The corrected average selling prices are present alongside daily demand, absolute weekly price changes, and percentage weekly price changes.

Missing price-change values at the beginning of each store-category series are expected because no preceding week is available for comparison.

The successful integration provides a consistent foundation for analyzing associations between pricing and retail demand and for subsequent machine learning feature engineering.

### Step 10.7 — Final Dataset Quality Check

At this stage, the forecasting dataset should contain the main information required for exploratory analysis and feature engineering:

* Store and state information
* Product category
* Date and calendar information
* Historical demand
* Holiday and event information
* SNAP availability
* Average selling price
* Weekly price changes

We will perform a final structural check before beginning exploratory data analysis.


In [41]:
# Step 10.7: Final dataset inspection

print("Dataset shape:", sales_long.shape)

print("\nColumns:")
print(sales_long.columns.tolist())

print("\nData types:")
print(sales_long.dtypes)

print("\nMissing values:")
display(
    sales_long.isna().sum().sort_values(ascending=False).head(15)
)

Dataset shape: (57390, 22)

Columns:
['store_id', 'state_id', 'cat_id', 'd', 'demand', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'avg_sell_price', 'price_change', 'price_change_pct']

Data types:
store_id                    object
state_id                    object
cat_id                      object
d                           object
demand                      uint16
date                datetime64[ns]
wm_yr_wk                     int64
weekday                     object
wday                         int64
month                        int64
year                         int64
event_name_1                object
event_type_1                object
event_name_2                object
event_type_2                object
snap_CA                      int64
snap_TX                      int64
snap_WI                      int64
snap                         int64
avg_sell_price      

event_type_2        57270
event_name_2        57270
event_name_1        52770
event_type_1        52770
price_change          210
price_change_pct      210
wm_yr_wk                0
weekday                 0
wday                    0
month                   0
year                    0
state_id                0
date                    0
demand                  0
d                       0
dtype: int64

#### Interpretation — Data Structure and Missing Values

The integrated dataset contains **57,390 daily store-category observations and 22 columns**, combining demand, calendar information, event indicators, SNAP availability, and pricing features.

The demand column contains no missing values, confirming that the target variable has been preserved during data preparation.

Missing event names and event types are expected because most calendar dates do not correspond to recorded events.

The 210 missing observations in `price_change` and `price_change_pct` correspond to the first seven days of each of the 30 store-category series, when no previous weekly selling price is available.

These missing values are structurally expected and should be handled appropriately during subsequent preprocessing rather than interpreted as data corruption.

Overall, the data structure and missing-value patterns are consistent with the intended daily store-category dataset.

In [42]:
# Step 10.7.1: Check for duplicates
print("Missing average prices:",
      sales_long["avg_sell_price"].isna().sum())

print("Missing demand:",
      sales_long["demand"].isna().sum())

print("Duplicate store-category-date rows:",
      sales_long.duplicated(
          ["store_id", "cat_id", "date"]
      ).sum())

Missing average prices: 0
Missing demand: 0
Duplicate store-category-date rows: 0


#### Interpretation — Data Integrity Validation

The final validation checks confirmed that the integrated dataset contains no missing average selling prices, no missing demand values, and no duplicate store-category-date observations.

These results demonstrate that the corrected weekly price aggregation was successfully integrated without compromising the completeness or uniqueness of the daily demand records.

The dataset is therefore consistent with the expected structure of 57,390 daily observations across 30 store-category combinations.

### Step 10.8 — Final Data Quality Audit

Before exporting the prepared dataset, a final data-quality audit is performed.

The purpose of this audit is to verify that:

* each store-category-day combination is unique;
* demand contains no negative values;
* selling prices contain no invalid values;
* all dates are valid;
* all expected stores and categories are present;
* the dataset contains no unexpected duplicate observations;
* missing values are understood rather than automatically removed.

At this stage, missing event values are expected because most calendar days do not contain an event. Missing price-change values also require investigation because they occur when a previous weekly price is unavailable.


In [43]:
# Step 10.8: Final data quality audit

print("========== DATA QUALITY AUDIT ==========\n")

# 1. Dataset dimensions
print("Dataset shape:", sales_long.shape)

# 2. Expected number of observations
expected_rows = 10 * 3 * 1913

print("Expected observations:", expected_rows)
print("Actual observations:", len(sales_long))

# 3. Duplicate store-category-day combinations
duplicates = sales_long.duplicated(
    subset=["store_id", "cat_id", "date"]
).sum()

print("\nDuplicate store-category-day rows:", duplicates)

# 4. Negative demand
negative_demand = (sales_long["demand"] < 0).sum()

print("Negative demand values:", negative_demand)

# 5. Invalid prices
invalid_prices = (
    sales_long["avg_sell_price"] <= 0
).sum()

print("Invalid price values:", invalid_prices)

# 6. Missing dates
print("Missing dates:", sales_long["date"].isna().sum())

# 7. Number of stores
print("Number of stores:", sales_long["store_id"].nunique())

# 8. Number of states
print("Number of states:", sales_long["state_id"].nunique())

# 9. Number of categories
print("Number of categories:", sales_long["cat_id"].nunique())

# 10. Date range
print(
    "\nDate range:",
    sales_long["date"].min(),
    "to",
    sales_long["date"].max()
)

========== DATA QUALITY AUDIT ==========

Dataset shape: (57390, 22)
Expected observations: 57390
Actual observations: 57390

Duplicate store-category-day rows: 0
Negative demand values: 0
Invalid price values: 0
Missing dates: 0
Number of stores: 10
Number of states: 3
Number of categories: 3

Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00


#### Interpretation — Final Data Quality Audit

The data quality audit confirmed that the prepared dataset contains **57,390 observations**, matching the expected number of daily store-category records.

No duplicate records, negative demand values, invalid selling prices, or missing dates were detected.

The dataset covers **10 Walmart stores across 3 states and 3 product categories**, spanning January 29, 2011, to April 24, 2016.

These results indicate that the dataset satisfies the structural and numerical integrity checks performed during data preparation.

The corrected pricing features can therefore proceed to the remaining preprocessing and export checks before being used in exploratory data analysis and machine learning.

In [44]:
# 10.8.1 Columns
print(sales_long.columns.tolist())

['store_id', 'state_id', 'cat_id', 'd', 'demand', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'avg_sell_price', 'price_change', 'price_change_pct']


In [45]:
print("Dataset shape:", sales_long.shape)

print("\nCurrent columns:")
print(sales_long.columns.tolist())

print("\nPrice features available:")
print(price_aggregated.columns.tolist())

Dataset shape: (57390, 22)

Current columns:
['store_id', 'state_id', 'cat_id', 'd', 'demand', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'avg_sell_price', 'price_change', 'price_change_pct']

Price features available:
['store_id', 'cat_id', 'wm_yr_wk', 'avg_sell_price', 'price_change', 'price_change_pct']


### Step 10.9 — Investigate Missing Price Changes

The price-change variables contain 210 missing observations.

The first observation in each store-category series naturally has no previous week and therefore cannot have a price change. However, the number of missing observations is greater than the 30 expected first-week observations.

Therefore, these missing values will be investigated before deciding whether they should be retained, replaced, or handled during feature engineering.

No automatic imputation will be performed until the cause of the missing values is understood.


In [46]:
# Step 10.9: Investigate missing price changes

missing_price_changes = sales_long[
    sales_long["price_change"].isna()
][
    [
        "store_id",
        "cat_id",
        "date",
        "wm_yr_wk",
        "avg_sell_price",
        "price_change",
        "price_change_pct"
    ]
]

print(
    "Rows with missing price changes:",
    len(missing_price_changes)
)

display(
    missing_price_changes.head(50)
)

Rows with missing price changes: 210


,store_id,cat_id,date,wm_yr_wk,avg_sell_price,price_change,price_change_pct
0,CA_1,FOODS,2011-01-29,11101,3.149220,NaN,NaN
1,CA_1,HOBBIES,2011-01-29,11101,4.468814,NaN,NaN
2,CA_1,HOUSEHOLD,2011-01-29,11101,5.416099,NaN,NaN
3,CA_2,FOODS,2011-01-29,11101,2.876585,NaN,NaN
4,CA_2,HOBBIES,2011-01-29,11101,4.606471,NaN,NaN
5,CA_2,HOUSEHOLD,2011-01-29,11101,5.560200,NaN,NaN
6,CA_3,FOODS,2011-01-29,11101,3.041969,NaN,NaN
7,CA_3,HOBBIES,2011-01-29,11101,4.586119,NaN,NaN
8,CA_3,HOUSEHOLD,2011-01-29,11101,5.614755,NaN,NaN
9,CA_4,FOODS,2011-01-29,11101,2.953270,NaN,NaN


#### Interpretation — Investigation of Missing Price Changes

The investigation identified **210 observations with missing price-change values**.

These observations correspond to the first seven days of the 30 store-category time series, covering the initial reporting week of the dataset.

Because weekly price changes are calculated relative to the preceding week, the first week has no historical price against which to compare.

Therefore, these missing values are structurally expected rather than the result of incomplete or incorrectly merged pricing data.

The observations should be retained to preserve the historical demand records. The missing price-change features can be handled during subsequent preprocessing, with the treatment documented.

### Step 10.10 — Standardize Final Column Names

To improve readability and consistency, the final prepared dataset uses descriptive column names following the `snake_case` convention.

The original M5 identifier `wm_yr_wk` **(Walmart Year-Week)** is retained because it is used to connect weekly price information with the calendar.

The less descriptive M5 fields `d` and `cat_id` and wday are renamed to `day_id`, `category` and `weekday_num` for readability in the final analytical dataset.


In [47]:
# Step 10.10: Standardize final column names

sales_long = sales_long.rename(
    columns={
        "d": "day_id",
        "cat_id": "category",
        "wday": "weekday_num"
    }
)

print("Final column names:")
print(sales_long.columns.tolist())

Final column names:
['store_id', 'state_id', 'category', 'day_id', 'demand', 'date', 'wm_yr_wk', 'weekday', 'weekday_num', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'avg_sell_price', 'price_change', 'price_change_pct']


#### Interpretation — Standardizing Column Names

The dataset columns were standardized to improve readability, consistency, and usability in subsequent analyses.

The original category, day identifier, and numerical weekday columns were renamed to `category`, `day_id`, and `weekday_num`, respectively.

All 22 columns were retained, including the corrected average selling prices and weekly price-change features.

This standardized structure provides a consistent foundation for exploratory data analysis, time series forecasting, and dashboard development.

### Step 10.11 — Remove Redundant SNAP Columns

The original M5 calendar contains three state-specific SNAP indicators:

* `snap_CA`
* `snap_TX`
* `snap_WI`

A unified `snap` feature was previously created by selecting the appropriate indicator according to each store's state.

Because the unified feature contains the relevant SNAP information for each observation, the three original state-specific columns are no longer required in the final analytical dataset.

They are removed to reduce redundancy while retaining the information needed for analysis and modeling.


In [48]:
# Step 10.11: Remove redundant SNAP columns

sales_long = sales_long.drop(
    columns=["snap_CA", "snap_TX", "snap_WI"]
)

print("Remaining SNAP feature:")
print("snap" in sales_long.columns)

print("\nDataset shape:", sales_long.shape)

Remaining SNAP feature:
True

Dataset shape: (57390, 19)


#### Interpretation — SNAP Feature Consolidation

The three state-specific SNAP indicators (`snap_CA`, `snap_TX`, and `snap_WI`) were removed after creating the consolidated `snap` feature.

The consolidated indicator retains the relevant SNAP information for each store based on its state, reducing redundant columns while preserving information useful for demand forecasting.

The resulting dataset contains **57,390 observations and 19 columns**.

This simplification improves feature consistency and prepares the dataset for exploratory data analysis and machine learning.

### Step 10.12 — Sort the Final Dataset

The final dataset is sorted chronologically within each store-category combination.

This ordering is important because the next stages of the project will create time-series features such as lagged demand and rolling averages.

Sorting before exporting also makes the CSV easier to inspect and reproduce.


In [49]:
# Step 10.12: Sort final dataset

sales_long = (
    sales_long
    .sort_values(
        ["store_id", "category", "date"]
    )
    .reset_index(drop=True)
)

display(sales_long.head(10))

,store_id,state_id,category,day_id,demand,date,wm_yr_wk,weekday,weekday_num,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap,avg_sell_price,price_change,price_change_pct
0,CA_1,CA,FOODS,d_1,3239,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,3.149220,NaN,NaN
1,CA_1,CA,FOODS,d_2,3137,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,3.149220,NaN,NaN
2,CA_1,CA,FOODS,d_3,2008,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,3.149220,NaN,NaN
3,CA_1,CA,FOODS,d_4,2258,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,3.149220,NaN,NaN
4,CA_1,CA,FOODS,d_5,2032,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,3.149220,NaN,NaN
5,CA_1,CA,FOODS,d_6,2407,2011-02-03,11101,Thursday,6,2,2011,NaN,NaN,NaN,NaN,1,3.149220,NaN,NaN
6,CA_1,CA,FOODS,d_7,2693,2011-02-04,11101,Friday,7,2,2011,NaN,NaN,NaN,NaN,1,3.149220,NaN,NaN
7,CA_1,CA,FOODS,d_8,3962,2011-02-05,11102,Saturday,1,2,2011,NaN,NaN,NaN,NaN,1,3.153394,0.004174,0.001325
8,CA_1,CA,FOODS,d_9,3418,2011-02-06,11102,Sunday,2,2,2011,SuperBowl,Sporting,NaN,NaN,1,3.153394,0.004174,0.001325
9,CA_1,CA,FOODS,d_10,2446,2011-02-07,11102,Monday,3,2,2011,NaN,NaN,NaN,NaN,1,3.153394,0.004174,0.001325


#### Interpretation — Final Dataset Sorting

The cleaned dataset was sorted by store, product category, and date to ensure chronological consistency within each time series.

This ordering is essential for time series forecasting because lag features, rolling statistics, and historical demand patterns depend on the correct sequence of observations.

The output confirms that daily demand, calendar information, SNAP indicators, and corrected pricing features remain aligned after sorting.

The dataset is now organized for subsequent validation, exploratory analysis, and predictive modeling.

### Step 10.13 — Final Cleaning Check

A final validation is performed after all cleaning and transformation steps.

The dataset should contain:

* 57,390 observations;
* 10 stores;
* 3 states;
* 3 product categories;
* one observation per store-category-day;
* no duplicate observations;
* no missing dates;
* no negative demand;
* no invalid prices;
* complete average price coverage;
* expected missing event values;
* structurally missing first-week price-change values.

If these checks pass, the dataset is ready to be exported and used by the EDA notebook.


In [50]:
# Step 10.13: Final cleaning validation

print("========== FINAL CLEANING CHECK ==========\n")

print("Shape:", sales_long.shape)

print("Duplicate rows:",
      sales_long.duplicated().sum())

print(
    "Duplicate store-category-day combinations:",
    sales_long.duplicated(
        subset=["store_id", "category", "date"]
    ).sum()
)

print(
    "Negative demand:",
    (sales_long["demand"] < 0).sum()
)

print(
    "Invalid prices:",
    (sales_long["avg_sell_price"] <= 0).sum()
)

print(
    "Missing dates:",
    sales_long["date"].isna().sum()
)

print(
    "Missing average prices:",
    sales_long["avg_sell_price"].isna().sum()
)

print(
    "Number of stores:",
    sales_long["store_id"].nunique()
)

print(
    "Number of states:",
    sales_long["state_id"].nunique()
)

print(
    "Number of categories:",
    sales_long["category"].nunique()
)

print(
    "Date range:",
    sales_long["date"].min(),
    "to",
    sales_long["date"].max()
)

========== FINAL CLEANING CHECK ==========

Shape: (57390, 19)
Duplicate rows: 0
Duplicate store-category-day combinations: 0
Negative demand: 0
Invalid prices: 0
Missing dates: 0
Missing average prices: 0
Number of stores: 10
Number of states: 3
Number of categories: 3
Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00


#### Interpretation — Final Data Cleaning Validation

The final data quality assessment confirmed that the cleaned M5 Walmart dataset contains **57,390 observations and 19 features**, covering 10 stores, 3 states, and 3 product categories.

No duplicate records, negative demand values, invalid selling prices, or missing dates were identified. Average selling prices are also complete.

The dataset covers the period from January 29, 2011, to April 24, 2016, providing a consistent historical foundation for time series analysis and demand forecasting.

The remaining missing values in the initial weekly price-change observations are expected and will be addressed appropriately during subsequent preprocessing.

Overall, the dataset has passed the final structural and numerical quality checks and is ready for the export stage.

### Step 10.14 — Export the Cleaned Dataset

The data-cleaning and preparation stage is now complete.

The resulting dataset contains aggregated daily retail demand at the **store-category-day level**, enriched with calendar, event, SNAP, and pricing information.

The cleaned dataset is exported separately from the original M5 files so that the raw data remains unchanged.

This file will be used as the input dataset for the EDA and machine-learning notebooks.


In [51]:
# Step 10.14: Export cleaned dataset

import os

# Create output directory
os.makedirs("../outputs", exist_ok=True)

# Final file path
cleaned_file = "../outputs/m5_cleaned_store_category_daily.csv"

# Export
sales_long.to_csv(
    cleaned_file,
    index=False
)

print("Cleaned dataset exported successfully.")
print("File:", cleaned_file)
print("Shape:", sales_long.shape)

Cleaned dataset exported successfully.
File: ../outputs/m5_cleaned_store_category_daily.csv
Shape: (57390, 19)


#### Interpretation — Exporting the Cleaned Dataset

The cleaned and validated M5 Walmart dataset was successfully exported as `m5_cleaned_store_category_daily.csv`.

The final dataset contains **57,390 daily observations and 19 columns**, representing 30 store-category time series across 10 stores and 3 product categories.

It includes historical demand, calendar information, event indicators, SNAP availability, and corrected weekly pricing features.

The exported dataset provides a consistent input for exploratory data analysis, time series forecasting, and subsequent dashboard development.

## Conclusion — Data Cleaning and Preparation

The data cleaning and preparation phase successfully transformed the original **Walmart M5 Forecasting dataset** into a structured, validated, and analysis-ready dataset suitable for exploratory data analysis and machine learning-based demand forecasting.

Given the large size of the original sales and pricing datasets, representative samples were initially examined to understand their structure, variables, and data types. The complete datasets were subsequently processed using **memory-efficient, chunk-based aggregation**, ensuring that the full historical information was retained without exceeding available memory resources.

Daily product-level sales were aggregated to the **store × category × day** level, producing **57,390 observations across 30 time series**, representing 10 stores, 3 states, 3 product categories, and 1,913 historical days (January 29, 2011, to April 24, 2016).

The dataset was enriched with calendar features, including weekdays, months, years, holidays, and special events. State-specific SNAP indicators were consolidated into a single `snap` variable, preserving the relevant benefit availability information for each store.

Additionally, product-level selling prices were aggregated into **weekly store-category average prices** using a weighted aggregation approach based on price observation counts. Absolute and percentage weekly price changes were then calculated to capture historical pricing movements and support subsequent demand analysis and forecasting.

Comprehensive data quality checks confirmed that the final dataset contains **no duplicate observations, negative demand values, invalid selling prices, missing dates, or missing average prices**. Missing event information reflects dates without recorded events, while missing price-change values during the first week of each store-category series are structurally expected because no preceding weekly price is available.

The final cleaned dataset contains **57,390 observations and 19 columns** and was successfully exported as `m5_cleaned_store_category_daily.csv`.

Overall, this phase established a **consistent and reproducible data foundation** for the remaining stages of the project. The next phase, **Exploratory Data Analysis and Time Series Analysis**, will investigate historical demand trends, seasonality, geographical and category-level differences, holiday and event patterns, SNAP-related demand variations, and associations between pricing and demand. These findings will guide feature engineering, model selection, and the development of reliable retail demand forecasting models.